In [1]:
import ROOT

# ----------------------------
# Open ROOT file
# ----------------------------
f = ROOT.TFile.Open("tumor.root")
if not f or f.IsZombie():
    print("Error: cannot open tumor.root")
    raise SystemExit

t = f.Get("t")
if not t:
    print("Error: tree 't' not found")
    f.ls()
    raise SystemExit

print("Entries =", t.GetEntries())

# ----------------------------
# Settings
# ----------------------------
phantom_vlm = 2   # change if needed
tumor_vlm   = 3   # change if needed

xmin, xmax = -10, 10
ymin, ymax = -10, 10
zslice = 1.0      # center slice thickness: abs(z) < zslice

nbins = 250

ROOT.gStyle.SetOptStat(0)

# =========================================================
# 1) Combined image: phantom + tumor together
# =========================================================
c1 = ROOT.TCanvas("c1", "Combined image", 900, 700)

h_combined = ROOT.TH2F(
    "h_combined",
    "Combined Phantom + Tumor; x; y",
    nbins, xmin, xmax,
    nbins, ymin, ymax
)

# Fill with deposited energy for both phantom and tumor
for i in range(t.GetEntries()):
    t.GetEntry(i)
    if t.vlm == phantom_vlm or t.vlm == tumor_vlm:
        h_combined.Fill(t.x, t.y, t.de)

h_combined.Draw("colz")
c1.SaveAs("combined_xy.png")

# =========================================================
# 2) Center slice image: phantom + tumor, only near z=0
# =========================================================
c2 = ROOT.TCanvas("c2", "Center slice", 900, 700)

h_slice = ROOT.TH2F(
    "h_slice",
    f"Center Slice |z| < {zslice}; x; y",
    nbins, xmin, xmax,
    nbins, ymin, ymax
)

for i in range(t.GetEntries()):
    t.GetEntry(i)
    if (t.vlm == phantom_vlm or t.vlm == tumor_vlm) and abs(t.z) < zslice:
        h_slice.Fill(t.x, t.y, t.de)

h_slice.Draw("colz")
c2.SaveAs("center_slice_xy.png")

print("Saved:")
print("  combined_xy.png")
print("  center_slice_xy.png")

Entries = 50000


Info in <TCanvas::Print>: png file combined_xy.png has been created


Saved:
  combined_xy.png
  center_slice_xy.png


Info in <TCanvas::Print>: png file center_slice_xy.png has been created
